In [73]:
# Importing libraries
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

# ── aesthetic era ──────────────────────────────────────────────────────────
# keeping it clean, no chartjunk, no trauma
plt.rcParams.update({
    "figure.facecolor"  : "white",
    "axes.facecolor"    : "#f9f9f9",
    "axes.edgecolor"    : "#d0d0d0",
    "axes.grid"         : True,
    "grid.color"        : "#e8e8e8",
    "grid.linestyle"    : "--",
    "grid.linewidth"    : 0.6,
    "font.family"       : "DejaVu Sans",
    "font.size"         : 11,
    "axes.titlesize"    : 12,
    "axes.titleweight"  : "bold",
    "axes.labelsize"    : 10,
    "xtick.labelsize"   : 9,
    "ytick.labelsize"   : 9,
    "legend.fontsize"   : 9,
    "figure.dpi"        : 130,
})

# brand colors — consistent throughout, no switching vibes mid-analysis
CLR_NORMAL  = "#3266ad"   # blue  — normal / good readings
CLR_ANOMALY = "#e24b4a"   # red   — something's off, pay attention
CLR_ROLLING = "#e8910a"   # amber — rolling mean / trend
CLR_GREY    = "#aaaaaa"   # grey  — background / reference lines
CLR_GREEN   = "#3b6d11"   # green — efficiency is thriving

TARGET = "Evaporator Leaving Water Temperature (°F)"
print("✅  imports loaded, we're built different")

✅  imports loaded, we're built different


In [74]:
# Loading the dataset
df_raw = pd.read_excel("CHILLER-1_Reports_26.XLSX", sheet_name="CHILLER-1INTEGRATIONREPORT0", header=None)

headers        = df_raw.iloc[3].tolist()
df_raw         = df_raw.iloc[4:].reset_index(drop=True)
df_raw.columns = [str(c).strip() for c in headers]

print(df_raw.shape)
print(df_raw.columns.tolist())
df_raw.head(3)

(4177, 32)
['DateTime', 'Active Cool/Heat Setpoint Temperature (°F)', 'Starter Motor Current L3 % RLA (%)', 'Active Current Limit Setpoint (%)', 'Actual Running Capacity (%)', 'AFD Output Power (kW)', 'Compressor Oil Pressure (psi)', 'Compressor Running Time ( )', 'Condenser Approach Temperature (^K)', 'Condenser Leaving Water Temperature (°F)', 'Condenser Entering Water Temperature (°F)', 'Condenser Saturated Rfgt Temp (°F)', 'Discharge Temperature (°F)', 'Evaporator Approach Temperature (^K)', 'Evaporator Entering Water Temperature (°F)', 'Evaporator Leaving Water Temperature (°F)', 'Evaporator Refrigerant Pressure (psi)', 'EXV Position Percent (%)', 'Frequency Command (Hz)', 'Starter Input Voltage AB (V)', 'Starter Input Voltage BC (V)', 'Starter Input Voltage CA (V)', 'Starter Motor Current L1 % RLA (%)', 'Starter Motor Current L1 (A)', 'Starter Motor Current L2 % RLA (%)', 'Starter Motor Current L2 (A)', 'Starter Motor Current L3 (A)', 'CHILLER-1 ACTIVE POWER (W) (kW)', 'CHILLER-1

,DateTime,Active Cool/Heat Setpoint Temperature (°F),Starter Motor Current L3 % RLA (%),Active Current Limit Setpoint (%),Actual Running Capacity (%),AFD Output Power (kW),Compressor Oil Pressure (psi),Compressor Running Time ( ),Condenser Approach Temperature (^K),Condenser Leaving Water Temperature (°F),...,Starter Motor Current L1 % RLA (%),Starter Motor Current L1 (A),Starter Motor Current L2 % RLA (%),Starter Motor Current L2 (A),Starter Motor Current L3 (A),CHILLER-1 ACTIVE POWER (W) (kW),CHILLER-1 AVG CURRENT (A),Compressor Starts ( ),COND Refrigerant Pressure (psi),Evaporator Saturated Rfgt Temp (°F)
0,2026-04-01 00:00:00,45,0,100,0,0,73.621162,15308360,0,81.716003,...,0,0,0,0,0,0,0,2758,73.621162,61.088364
1,2026-04-01 00:10:00,45,0,100,0,0,73.679176,15308360,0,81.697998,...,0,0,0,0,0,0,0,2758,73.650169,61.123329
2,2026-04-01 00:20:00,45,0,100,0,0,73.73719,15308360,0,81.697998,...,0,0,0,0,0,0,0,2758,73.679176,61.169949


In [75]:
# First check Run Status equivalent — Actual Running Capacity
print("Actual Running Capacity unique values:")
print(df_raw["Actual Running Capacity (%)"].value_counts().head(20))

print("\nAFD Output Power unique values:")
print(df_raw["AFD Output Power (kW)"].value_counts().head(10))

print("\nStarter Motor Current L1 sample:")
print(df_raw["Starter Motor Current L1 (A)"].value_counts().head(10))

Actual Running Capacity unique values:
Actual Running Capacity (%)
0            3155
5              32
45             13
1              12
90.800003      12
87.5            9
90.400002       9
81.699997       9
81.099998       9
90.099998       8
91.699997       8
91.099998       8
91.400002       8
91              8
88.400002       8
89.900002       7
91.5            7
87.099998       7
90.300003       7
90.5            7
Name: count, dtype: int64

AFD Output Power unique values:
AFD Output Power (kW)
0             3157
5               23
119.577499      14
1               14
154.035004      12
130.339996      11
87.254997       11
86.1875         10
88.339996       10
72.169998       10
Name: count, dtype: int64

Starter Motor Current L1 sample:
Starter Motor Current L1 (A)
0      3175
276      31
5        28
273      22
265      22
277      21
274      20
264      19
242      18
260      16
Name: count, dtype: int64


In [76]:
# Convert all columns to numeric
df_numeric = df_raw.copy()
df_numeric["DateTime"] = pd.to_datetime(df_numeric["DateTime"], format="mixed", errors="coerce")

for col in df_numeric.columns[1:]:
    df_numeric[col] = pd.to_numeric(df_numeric[col], errors="coerce")

# Filter running rows only — remove 0, 1, 5 (off/transient)
running = df_numeric[~df_numeric["Actual Running Capacity (%)"].isin([0, 1, 5])].reset_index(drop=True)

print(f"Total rows    : {len(df_numeric)}")
print(f"Running rows  : {len(running)}")
print(f"Date range    : {running['DateTime'].min()} → {running['DateTime'].max()}")

print(f"\n{'Column':<50} {'Min':>8} {'Max':>8} {'Mean':>8} {'Nulls':>6}")
print("-" * 80)
for col in running.columns[1:]:
    try:
        print(f"{col:<50} {running[col].min():>8.2f} "
              f"{running[col].max():>8.2f} "
              f"{running[col].mean():>8.2f} "
              f"{running[col].isnull().sum():>6}")
    except:
        pass

Total rows    : 4177
Running rows  : 978
Date range    : 2026-04-01 16:30:00 → 2026-04-29 22:10:00

Column                                                  Min      Max     Mean  Nulls
--------------------------------------------------------------------------------
Active Cool/Heat Setpoint Temperature (°F)             0.00 15512534.00 15907.99      0
Starter Motor Current L3 % RLA (%)                     0.00  2910.00    86.35      0
Active Current Limit Setpoint (%)                      0.00 15453520.00 15894.57      0
Actual Running Capacity (%)                            0.00 15830897.00 80132.73      0
AFD Output Power (kW)                                  0.00 15634430.00 47758.64      0
Compressor Oil Pressure (psi)                          0.00 15665925.00 63944.48      0
Compressor Running Time ( )                            0.00 15857055.00 13976354.79      0
Condenser Approach Temperature (^K)                    0.00 15764886.00 79745.83      0
Condenser Leaving Water Temper

In [77]:
# ── Step 1: understand what's happening ───────────────────────────────────
print("CORRUPTION DIAGNOSIS")
print("=" * 60)

# Physical limits for a chiller (domain knowledge)
PHYSICAL_LIMITS = {
    "Temperature (°F)"        : (30,   130),
    "Pressure (psi)"          : (0,    300),
    "Current % RLA (%)"       : (0,    150),
    "Current (A)"             : (0,    500),
    "Capacity (%)"            : (0,    120),
    "Power (kW)"              : (0,   2000),
    "Voltage (V)"             : (300,  500),
    "Frequency (Hz)"          : (0,     70),
    "Position Percent (%)"    : (0,    100),
    "Approach Temperature (^K)": (0,    20),
    "Running Time ( )"        : (0,    1e8),
    "Starts ( )"              : (0,    1e5),
}

def get_limit(col_name):
    for keyword, limits in PHYSICAL_LIMITS.items():
        if keyword.lower() in col_name.lower():
            return limits
    return None

for col in running.columns[1:]:
    limit = get_limit(col)
    if limit is None:
        continue
    lo, hi = limit
    n_bad = ((running[col] < lo) | (running[col] > hi)).sum()
    pct   = 100 * n_bad / len(running)
    flag  = "  ⚠️  CORRUPTED" if pct > 1 else ""
    print(f"  {col[:50]:<52} bad={n_bad:>4} ({pct:5.1f}%){flag}")

CORRUPTION DIAGNOSIS
  Active Cool/Heat Setpoint Temperature (°F)           bad=  75 (  7.7%)  ⚠️  CORRUPTED
  Actual Running Capacity (%)                          bad=  26 (  2.7%)  ⚠️  CORRUPTED
  AFD Output Power (kW)                                bad=   5 (  0.5%)
  Compressor Oil Pressure (psi)                        bad=  11 (  1.1%)  ⚠️  CORRUPTED
  Compressor Running Time ( )                          bad=   0 (  0.0%)
  Condenser Approach Temperature (^K)                  bad=  37 (  3.8%)  ⚠️  CORRUPTED
  Condenser Leaving Water Temperature (°F)             bad=  62 (  6.3%)  ⚠️  CORRUPTED
  Condenser Entering Water Temperature (°F)            bad=  80 (  8.2%)  ⚠️  CORRUPTED
  Discharge Temperature (°F)                           bad=  72 (  7.4%)  ⚠️  CORRUPTED
  Evaporator Approach Temperature (^K)                 bad=  28 (  2.9%)  ⚠️  CORRUPTED
  Evaporator Entering Water Temperature (°F)           bad=  76 (  7.8%)  ⚠️  CORRUPTED
  Evaporator Leaving Water Temperature (°

In [78]:
# ── what happened: BMS register overflow ──────────────────────────────────
# when a sensor drops offline the SCADA register returns its max int value
# (looks like 15,512,534 °F) — these are NOT real readings, they're bit-flips
# we null them out and interpolate back using time-aware method

COLUMN_LIMITS = {
    "Active Cool/Heat Setpoint Temperature (°F)"   : (30,   90),
    "Actual Running Capacity (%)"                  : (5,   110),
    "AFD Output Power (kW)"                        : (0,  2000),
    "Compressor Oil Pressure (psi)"                : (0,   300),
    "Compressor Running Time ( )"                  : (0,    2e7),
    "Condenser Approach Temperature (^K)"          : (0,    20),
    "Condenser Leaving Water Temperature (°F)"     : (60,  110),
    "Condenser Entering Water Temperature (°F)"    : (60,  110),
    "Condenser Saturated Rfgt Temp (°F)"           : (60,  120),
    "Discharge Temperature (°F)"                   : (70,  200),
    "Evaporator Approach Temperature (^K)"         : (0,    20),
    "Evaporator Entering Water Temperature (°F)"   : (38,   80),
    "Evaporator Leaving Water Temperature (°F)"    : (38,   75),
    "Evaporator Refrigerant Pressure (psi)"        : (30,  200),
    "EXV Position Percent (%)"                     : (0,   100),
    "Frequency Command (Hz)"                       : (0,    70),
    "Starter Motor Current L1 % RLA (%)"           : (0,   150),
    "Starter Motor Current L2 % RLA (%)"           : (0,   150),
    "Starter Motor Current L3 % RLA (%)"           : (0,   150),
    "Starter Motor Current L1 (A)"                 : (0,   500),
    "Starter Motor Current L2 (A)"                 : (0,   500),
    "Starter Motor Current L3 (A)"                 : (0,   500),
    "CHILLER-1 ACTIVE POWER (W) (kW)"              : (0,  2000),
    "CHILLER-1 AVG CURRENT (A)"                    : (0,   500),
    "Compressor Starts ( )"                        : (0,   1e5),
    "COND Refrigerant Pressure (psi)"              : (0,   300),
    "Evaporator Saturated Rfgt Temp (°F)"          : (30,   80),
}

df_clean = running.copy()
replaced_log = {}

# step 1 — null out impossible values
for col, (lo, hi) in COLUMN_LIMITS.items():
    if col not in df_clean.columns: continue
    mask = (df_clean[col] < lo) | (df_clean[col] > hi)
    replaced_log[col] = int(mask.sum())
    df_clean.loc[mask, col] = np.nan

# step 2 — time-aware interpolation (readings are 10-min apart, limit=3 = 30 min max gap)
df_clean = df_clean.set_index("DateTime").sort_index()
df_clean = df_clean.interpolate(method="time", limit=3)
df_clean = df_clean.reset_index()

# step 3 — anything still NaN → column median (safe fallback, no drama)
for col in df_clean.select_dtypes("number").columns:
    df_clean[col].fillna(df_clean[col].median(), inplace=True)

total_fixed = sum(replaced_log.values())
print(f" corrupted values fixed: {total_fixed} total")
print(f"    remaining nulls: {df_clean.select_dtypes('number').isnull().sum().sum()} (should be 0)")
print(f"\n  top offenders (the ones that needed the most fixing):")
for col, n in sorted(replaced_log.items(), key=lambda x: -x[1]):
    if n > 0:
        print(f"    {col[:55]:<57} fixed={n}")

 corrupted values fixed: 809 total
    remaining nulls: 138 (should be 0)

  top offenders (the ones that needed the most fixing):
    Condenser Saturated Rfgt Temp (°F)                        fixed=89
    Evaporator Entering Water Temperature (°F)                fixed=88
    Evaporator Leaving Water Temperature (°F)                 fixed=85
    Condenser Entering Water Temperature (°F)                 fixed=82
    Discharge Temperature (°F)                                fixed=80
    Active Cool/Heat Setpoint Temperature (°F)                fixed=76
    Condenser Leaving Water Temperature (°F)                  fixed=74
    Evaporator Refrigerant Pressure (psi)                     fixed=65
    Condenser Approach Temperature (^K)                       fixed=37
    Actual Running Capacity (%)                               fixed=30
    Evaporator Approach Temperature (^K)                      fixed=28
    Frequency Command (Hz)                                    fixed=15
    Compressor Oi

In [79]:
# ── find exactly where the nulls are and why interpolation missed them ─────
# if a sensor was offline for >30 min straight, limit=3 isn't enough to bridge it
# we need to know the gap lengths before deciding the fix strategy

null_summary = []
for col in df_clean.select_dtypes("number").columns:
    n_null = df_clean[col].isnull().sum()
    if n_null == 0:
        continue

    # find consecutive null runs — how long is each gap?
    is_null  = df_clean[col].isnull()
    run_lens = []
    count    = 0
    for v in is_null:
        if v:
            count += 1
        else:
            if count > 0:
                run_lens.append(count)
                count = 0
    if count > 0:
        run_lens.append(count)

    null_summary.append({
        "column"      : col,
        "total_nulls" : n_null,
        "n_gaps"      : len(run_lens),
        "max_gap_len" : max(run_lens) if run_lens else 0,
        "avg_gap_len" : round(np.mean(run_lens), 1) if run_lens else 0,
    })

null_df = pd.DataFrame(null_summary).sort_values("total_nulls", ascending=False)

print("REMAINING NULL DIAGNOSIS")
print("=" * 72)
print(f"  {'Column':<52} {'nulls':>6}  {'gaps':>5}  {'max gap':>8}  {'avg gap':>8}")
print("  " + "-" * 72)
for _, row in null_df.iterrows():
    gap_flag = "  ⚠️  long gap" if row["max_gap_len"] > 6 else ""
    print(f"  {row['column'][:52]:<52} {row['total_nulls']:>6}  "
          f"{row['n_gaps']:>5}  {row['max_gap_len']:>8}  "
          f"{row['avg_gap_len']:>8}{gap_flag}")

print(f"\n  total nulls remaining : {null_df['total_nulls'].sum()}")
print(f"  columns affected      : {len(null_df)}")
print(f"\n  → max gap of {null_df['max_gap_len'].max()} readings = "
      f"{null_df['max_gap_len'].max() * 10} min offline")
print(f"  → our limit=3 only covers 30 min — that's why these weren't caught")

REMAINING NULL DIAGNOSIS
  Column                                                nulls   gaps   max gap   avg gap
  ------------------------------------------------------------------------
  Evaporator Entering Water Temperature (°F)               33      8         9       4.1  ⚠️  long gap
  Condenser Saturated Rfgt Temp (°F)                       28      8         9       3.5  ⚠️  long gap
  Active Cool/Heat Setpoint Temperature (°F)               20      7         8       2.9  ⚠️  long gap
  Discharge Temperature (°F)                               17      7         4       2.4
  Evaporator Leaving Water Temperature (°F)                14      8         5       1.8
  Condenser Entering Water Temperature (°F)                12      6         4       2.0
  Condenser Leaving Water Temperature (°F)                  5      3         2       1.7
  Starter Motor Current L2 % RLA (%)                        2      1         2       2.0
  Condenser Approach Temperature (^K)                    

In [80]:
# ── root cause confirmed: max gap = 9 readings = 90 min offline ───────────
# our limit=3 only bridged 30 min — anything longer got abandoned as NaN
# fix: stage 1 removes the cap entirely (safe — chiller temps drift slowly)
#      stage 2 ffill+bfill catches start/end of series edge cases
#      stage 3 median is the absolute last resort — should hit 0 values

print("THREE-STAGE NULL FILL")
print("=" * 55)

df_temp = df_clean.copy().set_index("DateTime").sort_index()

# ── stage 1: unlimited time-aware interpolation ───────────────────────────
# linear bridge between last known good value and next known good value
# 90 min on a temperature sensor = safe to interpolate (slow-changing physics)
nulls_before = df_temp.select_dtypes("number").isnull().sum().sum()
df_temp = df_temp.interpolate(method="time", limit=None)
nulls_after_s1 = df_temp.select_dtypes("number").isnull().sum().sum()
print(f"  stage 1 (time interpolation) : {nulls_before} → {nulls_after_s1} nulls")

# ── stage 2: forward fill then backward fill ──────────────────────────────
# handles series edges — if the gap is at the very start (no left neighbor)
# or the very end (no right neighbor), interpolation can't help but ffill can
df_temp = df_temp.ffill().bfill()
nulls_after_s2 = df_temp.select_dtypes("number").isnull().sum().sum()
print(f"  stage 2 (ffill + bfill)      : {nulls_after_s1} → {nulls_after_s2} nulls")

# ── stage 3: median fill — last resort ───────────────────────────────────
stage3_used = []
for col in df_temp.select_dtypes("number").columns:
    remaining = df_temp[col].isnull().sum()
    if remaining > 0:
        med = df_temp[col].median()
        df_temp[col].fillna(med, inplace=True)
        stage3_used.append((col, remaining))

nulls_after_s3 = df_temp.select_dtypes("number").isnull().sum().sum()
print(f"  stage 3 (median fallback)    : {nulls_after_s2} → {nulls_after_s3} nulls")

if stage3_used:
    print(f"\n  columns that needed stage 3:")
    for col, n in stage3_used:
        print(f"    {col[:55]}  ({n} values)")
else:
    print(f"\n  stage 3 not needed — interpolation handled everything")

df_clean = df_temp.reset_index()

final_nulls = df_clean.select_dtypes("number").isnull().sum().sum()
print(f"\n{'✅' if final_nulls == 0 else '❌'}  final null count: {final_nulls}")

THREE-STAGE NULL FILL
  stage 1 (time interpolation) : 138 → 16 nulls
  stage 2 (ffill + bfill)      : 16 → 0 nulls
  stage 3 (median fallback)    : 0 → 0 nulls

  stage 3 not needed — interpolation handled everything

✅  final null count: 0


In [81]:
# ── this is the trust-but-verify step ─────────────────────────────────────
# interpolation can bridge two corrupt neighbors and produce a plausible-looking
# but wrong value — we catch that by checking min/mean/max vs physical limits
# anything outside limits after cleaning = the column needs a tighter bound
# in COLUMN_LIMITS back in cell 3

CHECK_COLS = [
    ("Evaporator Leaving Water Temperature (°F)",     38,   75),
    ("Evaporator Entering Water Temperature (°F)",    38,   80),
    ("Condenser Leaving Water Temperature (°F)",      60,  110),
    ("Condenser Entering Water Temperature (°F)",     60,  110),
    ("Condenser Saturated Rfgt Temp (°F)",            60,  120),
    ("Discharge Temperature (°F)",                    70,  200),
    ("Compressor Oil Pressure (psi)",                  0,  300),
    ("CHILLER-1 ACTIVE POWER (W) (kW)",                0, 2000),
    ("Actual Running Capacity (%)",                    5,  110),
    ("Evaporator Refrigerant Pressure (psi)",         30,  200),
    ("Active Cool/Heat Setpoint Temperature (°F)",    30,   90),
    ("Frequency Command (Hz)",                         0,   70),
    ("EXV Position Percent (%)",                       0,  100),
]

print("POST-FILL SANITY CHECK")
print("=" * 90)
print(f"  {'Column':<52} {'min':>7}  {'max':>7}  {'mean':>7}  {'nulls':>6}  status")
print("  " + "-" * 90)

issues = []
for col, lo, hi in CHECK_COLS:
    if col not in df_clean.columns:
        continue
    s     = df_clean[col]
    c_min = round(s.min(), 2)
    c_max = round(s.max(), 2)
    c_mean= round(s.mean(), 2)
    nulls = s.isnull().sum()

    ok = (c_min >= lo * 0.95) and (c_max <= hi * 1.05) and (nulls == 0)
    status = "ok" if ok else "⚠️  flag"
    if not ok:
        issues.append((col, c_min, c_max, lo, hi, nulls))

    print(f"  {col[:52]:<52} {c_min:>7.2f}  {c_max:>7.2f}  {c_mean:>7.2f}  {nulls:>6}  {status}")

print()
if not issues:
    print("  all columns clean — physically valid, zero nulls")
    print("      safe to proceed to z-score + correlation")
else:
    print("  ⚠️  flagged columns (need tighter limits in COLUMN_LIMITS):")
    for col, c_min, c_max, lo, hi, nulls in issues:
        reason = []
        if c_min < lo * 0.95 : reason.append(f"min {c_min} < floor {lo}")
        if c_max > hi * 1.05 : reason.append(f"max {c_max} > ceiling {hi}")
        if nulls > 0          : reason.append(f"{nulls} nulls remain")
        print(f"    • {col[:52]}: {', '.join(reason)}")
    print()
    print("  fix: go back to COLUMN_LIMITS in cell 3, tighten the (lo, hi)")
    print("       for flagged columns, re-run cell 3 → this cell")

POST-FILL SANITY CHECK
  Column                                                   min      max     mean   nulls  status
  ------------------------------------------------------------------------------------------
  Evaporator Leaving Water Temperature (°F)              42.64    74.52    47.11       0  ok
  Evaporator Entering Water Temperature (°F)             45.00    79.90    55.28       0  ok
  Condenser Leaving Water Temperature (°F)               62.13   100.38    87.42       0  ok
  Condenser Entering Water Temperature (°F)              61.08   100.00    79.93       0  ok
  Condenser Saturated Rfgt Temp (°F)                     61.02   119.77    87.67       0  ok
  Discharge Temperature (°F)                             75.54   124.95   101.53       0  ok
  Compressor Oil Pressure (psi)                           0.00   206.00   101.26       0  ok
  CHILLER-1 ACTIVE POWER (W) (kW)                         0.00   189.26   104.28       0  ok
  Actual Running Capacity (%)              

In [82]:
# ── now that nulls are gone, redo the core analysis ───────────────────────
# z-score on a series with NaN silently skips those rows — results were wrong before

TARGET    = "Evaporator Leaving Water Temperature (°F)"
THRESHOLD = 3.0

df_clean["z_score"]    = np.abs(stats.zscore(df_clean[TARGET].dropna()))
df_clean["is_anomaly"] = df_clean["z_score"] > THRESHOLD

n_total   = len(df_clean)
n_anomaly = int(df_clean["is_anomaly"].sum())

print(f"Z-SCORE RESULTS — fully clean data (0 nulls)")
print(f"  threshold : ±{THRESHOLD}σ")
print(f"  total     : {n_total}")
print(f"  normal    : {n_total - n_anomaly}  ({100*(n_total-n_anomaly)/n_total:.1f}%)")
print(f"  anomalies : {n_anomaly}  ({100*n_anomaly/n_total:.1f}%)")

# ── pearson on clean data ─────────────────────────────────────────────────
numeric_cols = (df_clean.select_dtypes("number")
                .columns.drop(["z_score","is_anomaly"], errors="ignore"))

results = []
for col in numeric_cols:
    if col == TARGET: continue
    tmp = df_clean[[TARGET, col]].dropna()
    if len(tmp) < 30 or tmp[col].std() == 0: continue
    r, p = stats.pearsonr(tmp[TARGET], tmp[col])
    results.append({"variable": col, "r": r, "p_value": p, "abs_r": abs(r)})

corr_df = (pd.DataFrame(results)
             .sort_values("abs_r", ascending=False)
             .reset_index(drop=True))

def strength(r):
    a = abs(r)
    if a >= .70: return "very strong"
    if a >= .50: return "strong"
    if a >= .30: return "moderate"
    if a >= .10: return "weak"
    return "negligible"

corr_df["strength"]    = corr_df["r"].apply(strength)
corr_df["significant"] = corr_df["p_value"] < 0.05

print(f"\nPEARSON r  —  {TARGET}")
print(f"  {'Variable':<52} {'r':>7}  {'Strength':<14} Sig")
print("  " + "-" * 78)
for _, row in corr_df.iterrows():
    bar = "█" * int(abs(row["r"]) * 18)
    sig = "✓" if row["significant"] else "✗"
    print(f"  {row['variable'][:52]:<52} {row['r']:>+7.4f}  "
          f"{row['strength']:<14} {sig}  {bar}")

Z-SCORE RESULTS — fully clean data (0 nulls)
  threshold : ±3.0σ
  total     : 978
  normal    : 923  (94.4%)
  anomalies : 55  (5.6%)

PEARSON r  —  Evaporator Leaving Water Temperature (°F)
  Variable                                                   r  Strength       Sig
  ------------------------------------------------------------------------------
  Evaporator Saturated Rfgt Temp (°F)                  +0.8760  very strong    ✓  ███████████████
  Evaporator Refrigerant Pressure (psi)                +0.8439  very strong    ✓  ███████████████
  Starter Motor Current L3 (A)                         -0.6794  strong         ✓  ████████████
  Starter Motor Current L1 (A)                         -0.6670  strong         ✓  ████████████
  Starter Motor Current L2 % RLA (%)                   -0.6344  strong         ✓  ███████████
  Starter Motor Current L3 % RLA (%)                   -0.6141  strong         ✓  ███████████
  Compressor Running Time ( )                          -0.6131  strong